In [2]:
# UPSAMPLE HEALTHY: safe synthetic generation (copy into a Jupyter cell and run)
import os
import numpy as np
import pandas as pd
from sklearn.utils import resample
from collections import Counter

# --- CONFIG: change path if your file is elsewhere ---
DATA_PATH = "expanded_medical_dataset_modified.xlsx"
OUT_PATH = "expanded_medical_dataset_balanced_for_user.xlsx"

# Columns (must match your dataset)
NUMERIC_COLS = ["Age","ESR","CRP","RF","C3","C4"]
CATEGORICAL_COLS = ["Gender","Anti-CCP","HLA-B27","ANA","Anti-Ro","Anti-La","Anti-dsDNA","Anti-Sm"]
LABEL_COL = "Disease"

# Load dataset
df = pd.read_excel(DATA_PATH)
print("Loaded rows:", len(df))

# Ensure required columns exist
missing_cols = [c for c in (NUMERIC_COLS + CATEGORICAL_COLS + [LABEL_COL]) if c not in df.columns]
if missing_cols:
    raise ValueError("Missing expected columns in dataset: " + ", ".join(missing_cols))

# Normalize label strings (but keep original exact healthy label if present)
# We'll detect the exact token used for healthy by looking for 'healthy' case-insensitively.
healthy_mask = df[LABEL_COL].astype(str).str.lower() == 'healthy'
if healthy_mask.sum() == 0:
    # fallback: if no exact 'healthy' label found, try to find label with smallest count and treat it as healthy candidate
    counts = df[LABEL_COL].value_counts()
    smallest_label = counts.idxmin()
    print("No explicit 'healthy' label found. Treating least frequent label as healthy:", smallest_label)
    healthy_label_token = smallest_label
else:
    # find the exact token used (first match)
    healthy_label_token = df.loc[healthy_mask, LABEL_COL].iloc[0]

# Separate healthy and others
df_healthy = df[df[LABEL_COL] == healthy_label_token].copy()
df_other = df[df[LABEL_COL] != healthy_label_token].copy()

print("\nOriginal class counts:")
print(df[LABEL_COL].value_counts())

n_healthy = len(df_healthy)
n_other = len(df_other)

# Compute target healthy count: mean of other class counts (safe compromise)
other_class_counts = df_other[LABEL_COL].value_counts()
if len(other_class_counts) == 0:
    target_healthy = n_healthy
else:
    target_healthy = int(other_class_counts.mean())
target_healthy = max(target_healthy, n_healthy)
n_needed = target_healthy - n_healthy

print(f"\nHealthy token used: '{healthy_label_token}'")
print("Healthy before:", n_healthy)
print("Target healthy:", target_healthy)
print("Healthy to add:", n_needed)

if n_needed <= 0:
    print("Healthy count already at/above target. Saving a copy of original dataset to output path.")
    df.to_excel(OUT_PATH, index=False)
else:
    # Create synthetic healthy samples by sampling existing healthy rows and perturbing numeric features slightly.
    rng = np.random.default_rng(42)
    sampled = resample(df_healthy, replace=True, n_samples=n_needed, random_state=42)
    synth_list = []
    for _, row in sampled.iterrows():
        new = row.copy()
        # Age: +/- 2 years noise
        try:
            age = float(new['Age'])
            age += int(rng.normal(0, 2))
            if age < 0: age = abs(age)
            new['Age'] = int(round(age))
        except Exception:
            pass
        # ESR, CRP: small relative noise ~N(0, 7%)
        for col in ['ESR', 'CRP']:
            try:
                v = float(new[col])
                if v <= 0:
                    v = max(0.0, v + rng.normal(0, 0.5))
                else:
                    v = v * (1.0 + float(rng.normal(0, 0.07)))
                new[col] = round(max(0.0, v), 3)
            except Exception:
                pass
        # RF: numeric perturbation if numeric
        try:
            v = float(new['RF'])
            if v <= 0:
                v = max(0.0, v + rng.normal(0, 0.5))
            else:
                v = v * (1.0 + float(rng.normal(0, 0.05)))
            new['RF'] = round(max(0.0, v), 3)
        except Exception:
            pass
        # C3, C4: small relative noise ~5%
        for col in ['C3', 'C4']:
            try:
                v = float(new[col])
                v = v * (1.0 + float(rng.normal(0, 0.05)))
                new[col] = round(max(0.0, v), 3)
            except Exception:
                pass
        # Categorical: keep same values from sampled row (preserves medical realism)
        new[LABEL_COL] = healthy_label_token
        synth_list.append(new)
    df_synth = pd.DataFrame(synth_list)
    df_balanced = pd.concat([df, df_synth], ignore_index=True)
    # Shuffle and save
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    df_balanced.to_excel(OUT_PATH, index=False)
    print("\nSaved balanced dataset to:", OUT_PATH)
    print("New class counts:")
    print(df_balanced[LABEL_COL].value_counts())

    # Show a few synthetic rows for verification
    display(df_synth.head(8))

# Final note
print("\nDone. If you want a stronger increase in healthy samples, change the target to other_class_counts.max() or a custom number.")


Loaded rows: 29043

Original class counts:
Disease
rheumatoid arthritis            12848
systemic lupus erythematosus    11355
healthy                          4840
Name: count, dtype: int64

Healthy token used: 'healthy'
Healthy before: 4840
Target healthy: 12101
Healthy to add: 7261

Saved balanced dataset to: expanded_medical_dataset_balanced_for_user.xlsx
New class counts:
Disease
rheumatoid arthritis            12848
healthy                         12101
systemic lupus erythematosus    11355
Name: count, dtype: int64


,Age,Gender,ESR,CRP,RF,Anti-CCP,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm,C3,C4,Disease
5010,20,Female,9.838,15.282,23.635,29.729764,Negative,Negative,Negative,Negative,Positive,Negative,53.826,4.194,healthy
22955,32,Male,13.268,7.195,11.584,9.895371,Negative,Negative,Positive,Negative,Positive,Positive,41.633,9.518,healthy
18677,34,Female,9.393,15.106,26.953,8.617935,Positive,Negative,Positive,Negative,Negative,Negative,119.221,19.440,healthy
2670,63,Male,8.468,0.000,13.794,37.507270,Positive,Positive,Negative,Negative,Negative,Positive,64.192,10.968,healthy
26672,45,Female,18.354,11.340,45.765,19.268641,Negative,Positive,Positive,Positive,Positive,Negative,99.675,0.000,healthy
20910,68,Female,36.847,13.885,32.577,34.762921,Negative,Negative,Positive,Positive,Positive,Negative,150.421,27.324,healthy
19252,42,Female,13.498,6.336,31.831,26.433887,Positive,Negative,Negative,Positive,Positive,Positive,78.330,6.739,healthy
17648,57,Female,2.072,1.474,22.078,16.249505,Negative,Positive,Negative,Negative,Positive,Negative,69.274,17.923,healthy



Done. If you want a stronger increase in healthy samples, change the target to other_class_counts.max() or a custom number.
